# Vehicle CO2 emissions regression

Original project by KwalaTheCoder. Modern AI-assisted refactor; see README.md and PROVENANCE.json.

Configure the dataset path below. No benchmark results are prefilled.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from mlproject.core import read_data, train, predict
CONFIG = json.loads(Path('project.json').read_text())
DATA = Path('data/dataset.csv')


## Inspect the schema

Confirm which variables are available at prediction time. Do not choose predictors using holdout correlations.

In [ ]:
df = read_data(DATA)
print(df.shape)
display(df.head())
display(df.isna().sum().to_frame('missing'))


## Select with training-fold CV; evaluate once on holdout

For repeated entities supply group_column. Do not tune after reading final holdout metrics.

In [ ]:
report = train(df, CONFIG, 'artifacts', folds=5, seed=42, tune=False)
display(report['leaderboard'])
display(report['holdout'])


## Predict with the saved pipeline

Use a separate input file with the documented raw predictors. No fitted preprocessing needs to be recreated.

In [ ]:
NEW_DATA = Path('data/new_rows.csv')
if NEW_DATA.exists():
    predictions = predict('artifacts/model.joblib', read_data(NEW_DATA))
    display(predictions.head())
    predictions.to_csv('predictions.csv', index=False)
